In [ ]:
# Enable CUDA GPU Acceleration - SELECT GPU T4 ×2 ON KAGGLE!
import os
import sys

# Check if GPU is available
IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    print("🚀 Setting up CUDA GPU acceleration...")
    
    # Clean up conflicting CuPy installations first
    print("📦 Cleaning up any existing CuPy installations...")
    !pip uninstall -y cupy-cuda11x cupy-cuda12x cupy 2>/dev/null || true
    
    # Install CuPy for CUDA 11.x (Kaggle uses CUDA 11.8)
    print("📦 Installing CuPy for CUDA 11.x...")
    !pip install -q cupy-cuda11x
    
    try:
        import cupy as cp
        
        # Get GPU info properly
        device = cp.cuda.Device()
        gpu_name = cp.cuda.runtime.getDeviceProperties(device.id)['name'].decode('utf-8')
        total_mem, free_mem = cp.cuda.Device().mem_info
        
        print(f"✓ CuPy installed! GPU: {gpu_name}")
        print(f"✓ GPU Memory: {total_mem / 1e9:.1f} GB total, {free_mem / 1e9:.1f} GB free")
        
        # Test GPU
        test = cp.array([1, 2, 3])
        result = cp.sum(test)
        print(f"✓ CUDA is working! Test sum: {result}")
        
        USE_GPU = True
    except Exception as e:
        print(f"⚠️ CuPy initialization failed: {e}")
        print("Falling back to CPU mode")
        USE_GPU = False
else:
    print("Running locally (CPU mode)")
    USE_GPU = False

print("="*60)

In [ ]:
# Setup environment and clone repository
import os
import sys

IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    print("Running on Kaggle")
    
    # Clone the repository if not already present
    if not os.path.exists('/kaggle/working/NLP_PROJECT_2025'):
        print("Cloning repository from mohab branch...")
        !git clone -b mohab https://github.com/MohabYasser2/NLP_PROJECT_2025.git /kaggle/working/NLP_PROJECT_2025
        print("✓ Repository cloned!")
    else:
        print("✓ Repository already exists, pulling latest changes...")
        !cd /kaggle/working/NLP_PROJECT_2025 && git fetch origin mohab && git reset --hard origin/mohab
        print("✓ Repository updated to latest version!")
    
    # CRITICAL: Delete old cached pickle files (they have wrong label counts)
    print("\n🗑️ Cleaning old cache files...")
    !rm -f /kaggle/working/*_processed.pkl
    print("✓ Cache cleared - will regenerate with fixed preprocessing")
    
    # Add to Python path
    sys.path.insert(0, '/kaggle/working/NLP_PROJECT_2025')
else:
    print("Running locally")
    # Add parent directory to path for local execution
    sys.path.insert(0, os.path.abspath('..'))

print(f"✓ Python path configured: {sys.path[0]}")

In [ ]:
# Detect available datasets on Kaggle
if IS_KAGGLE:
    print("\n=== Available Kaggle Input Datasets ===")
    !ls -la /kaggle/input/
    print("\n=== Looking for data files ===")
    import glob
    data_files = glob.glob('/kaggle/input/**/*.txt', recursive=True)
    print(f"Found .txt files: {data_files}")
    
    # Try to auto-detect the dataset directory
    if data_files:
        dataset_dir = os.path.dirname(data_files[0])
        print(f"\n✓ Auto-detected dataset directory: {dataset_dir}")
        
        # List all files in dataset directory
        all_files = [os.path.basename(f) for f in data_files if os.path.dirname(f) == dataset_dir]
        print(f"Available files: {all_files}")
        
        # Auto-detect file names
        TRAIN_FILE = os.path.join(dataset_dir, 'train.txt')
        
        # Check for dev/val file
        if 'val.txt' in all_files:
            DEV_FILE = os.path.join(dataset_dir, 'val.txt')
            print("Using 'val.txt' for validation")
        elif 'dev.txt' in all_files:
            DEV_FILE = os.path.join(dataset_dir, 'dev.txt')
            print("Using 'dev.txt' for validation")
        else:
            print("⚠ Warning: No val.txt or dev.txt found, using train.txt for validation")
            DEV_FILE = TRAIN_FILE
        
        TEST_FILE = os.path.join(dataset_dir, 'test.txt')
    else:
        print("\n⚠ No .txt files found! Please add your dataset to this notebook.")
        print("Click 'Add Data' → Search for your dataset → Add to notebook")
else:
    TRAIN_FILE = '../data/train.txt'
    DEV_FILE = '../data/val.txt'
    TEST_FILE = '../data/test.txt'

OUTPUT_FILE = '/kaggle/working/submission.csv' if IS_KAGGLE else 'submission.csv'

print(f"\n=== Final Configuration ===")
print(f"Train file: {TRAIN_FILE} (exists: {os.path.exists(TRAIN_FILE) if 'TRAIN_FILE' in locals() else 'N/A'})")
print(f"Dev file: {DEV_FILE} (exists: {os.path.exists(DEV_FILE) if 'DEV_FILE' in locals() else 'N/A'})")
print(f"Test file: {TEST_FILE} (exists: {os.path.exists(TEST_FILE) if 'TEST_FILE' in locals() else 'N/A'})")
print(f"Output file: {OUTPUT_FILE}")

## Import Training Module

All the heavy implementation is in the `/src` directory. This notebook just calls the training function.

In [ ]:
# Import training function
from src.training.train_logreg import run_logreg_training

print("✓ Imports successful")

In [ ]:
# Verify repository structure
import os
print("Checking repository structure...")
src_path = '/kaggle/working/NLP_PROJECT_2025/src' if IS_KAGGLE else '../src'
if os.path.exists(src_path):
    print(f"✓ src directory found")
    models_path = os.path.join(src_path, 'models')
    if os.path.exists(models_path):
        print(f"✓ src/models directory found")
        files = os.listdir(models_path)
        print(f"  Files in models/: {files}")
    else:
        print(f"✗ src/models directory NOT found!")
else:
    print(f"✗ src directory NOT found at {src_path}!")

## 🚀 Train Optimized Model

This will train with OPTIMIZED parameters for much better accuracy:
- **15,000 features** (vs 1,000 in old version)
- **5-gram support** (1-5 grams)
- **Window size 7** (±3 characters context)
- **200 iterations** with higher learning rate
- **Full 50k training dataset**

⏱️ **Expected time**: 2-4 hours on Kaggle
🎯 **Target DER**: 25-35% (vs current 64.71%)

**Leave this running overnight for best results!**

In [ ]:
import pickle
from src.preprocessing import prepare_dataset
from src.models.logreg_model import LogisticRegressionModel
from src.training.train_logreg import run_logreg_training
from pathlib import Path

# Check if pre-trained model exists
MODEL_PATH = '/kaggle/working/NLP_PROJECT_2025/models/logreg_model.pkl' if IS_KAGGLE else '../models/logreg_model.pkl'
MODEL_EXISTS = os.path.exists(MODEL_PATH)

if MODEL_EXISTS:
    print("=" * 70)
    print(" " * 15 + "🔄 USING PRE-TRAINED MODEL")
    print("=" * 70)
    print(f"\n✓ Found existing model at: {MODEL_PATH}")
    print("💡 Loading model and evaluating on dev set...")
    print("💡 To retrain, rename or delete the existing model file")
    print("=" * 70)
    print("\n")
    
    # Load pre-trained model
    with open(MODEL_PATH, 'rb') as f:
        model_data = pickle.load(f)
    
    if isinstance(model_data, dict):
        hyperparams = model_data.get('hyperparams', {})
        model = LogisticRegressionModel(
            learning_rate=hyperparams.get('learning_rate', 0.01),
            max_iter=hyperparams.get('max_iter', 500),
            regularization=hyperparams.get('regularization', 0.01),
            max_features=hyperparams.get('max_features', 5000),
            ngram_range=hyperparams.get('ngram_range', (1, 3)),
            batch_size=hyperparams.get('batch_size', 64)
        )
        model.weights = model_data['weights']
        model.bias = model_data['bias']
        model.vectorizer = model_data['vectorizer']
        model.label_to_idx = model_data['label_to_idx']
        model.idx_to_label = model_data['idx_to_label']
        model.is_fitted = True
    else:
        model = model_data
    
    # Load dev data and evaluate
    cache_dir = Path('/kaggle/working') if IS_KAGGLE else Path('../data')
    cache_file = cache_dir / 'val_processed.pkl'
    
    if cache_file.exists():
        with open(cache_file, 'rb') as f:
            data = pickle.load(f)
        dev_texts, dev_labels = data['texts'], data['labels']
    else:
        dev_texts, dev_labels = prepare_dataset(str(DEV_FILE), str(cache_file.with_suffix('')))
    
    print(f"Evaluating on {len(dev_texts)} dev sentences...")
    dev_metrics = model.evaluate(dev_texts, dev_labels, window_size=3)
    
    print("\n" + "=" * 70)
    print(" " * 15 + "📈 PRE-TRAINED MODEL RESULTS")
    print("=" * 70)
    print(f"\n✓ Accuracy: {dev_metrics['accuracy']:.4f} ({dev_metrics['accuracy']*100:.2f}%)")
    print(f"✓ DER: {dev_metrics['der']:.4f} ({dev_metrics['der']*100:.2f}%)")
    print(f"✓ Correct: {dev_metrics['correct']:,} / {dev_metrics['total']:,}")
    print("=" * 70)
    
else:
    print("=" * 70)
    print(" " * 15 + "🚀 OPTIMIZED TRAINING MODE")
    print("=" * 70)
    print("\n📋 Configuration:")
    print("   • Full dataset: 50k training samples")
    print("   • Features: 15,000 (TF-IDF with 1-5 grams)")
    print("   • Context: Window size 7 (±3 characters)")
    print("   • Training: 200 iterations")
    print("   • Batch size: 256")
    print("   • Learning rate: 0.05")
    print("   • Regularization: 0.0005")
    print("\n⏱️  Expected time: 2-4 hours")
    print("🎯 Target DER: 25-35% (70-75% accuracy)")
    print("\n💡 Leave this running and check back later!")
    print("=" * 70)
    print("\n")
    
    # Check if test file exists
    TEST_FILE_EXISTS = os.path.exists(TEST_FILE) if 'TEST_FILE' in locals() else False
    
    if not TEST_FILE_EXISTS:
        print("⚠ Test file not available yet - will evaluate on dev set only\n")
        TEST_FILE = None
    
    # Run optimized training with new default parameters
    run_logreg_training(
        train_file=TRAIN_FILE,
        dev_file=DEV_FILE,
        test_file=TEST_FILE,
        output_file=OUTPUT_FILE
        # Parameters are now defaults in the function:
        # window_size=7, learning_rate=0.05, max_iter=200,
        # regularization=5e-4, max_features=15000,
        # ngram_range=(1,5), batch_size=256
    )
    
    print("\n" + "=" * 70)
    print(" " * 20 + "✅ TRAINING COMPLETE!")
    print("=" * 70)
    print("\n📊 Check the 'Dev Set Results' above for your improved DER")
    print("💾 Model saved to: /kaggle/working/models/logreg_model.pkl")
    if TEST_FILE_EXISTS:
        print(f"📤 Predictions saved to: {OUTPUT_FILE}")
    print("=" * 70)

## Download Trained Model

Download the trained model to use locally or for submission.

In [ ]:
if IS_KAGGLE:
    print("📦 Preparing model for download...")
    
    # Check if model exists
    model_path = '/kaggle/working/models/logreg_model.pkl'
    if os.path.exists(model_path):
        # Get model size
        size_mb = os.path.getsize(model_path) / (1024 * 1024)
        print(f"✓ Model found: {size_mb:.2f} MB")
        print(f"✓ Model location: {model_path}")
        print("\n📥 To download:")
        print("   1. Go to the right sidebar → Output")
        print("   2. Find 'models/logreg_model.pkl'")
        print("   3. Click download icon")
        print("\n💡 Or use this in your local environment:")
        print(f"   from src.models.logreg_model import LogisticRegressionModel")
        print(f"   import pickle")
        print(f"   with open('logreg_model.pkl', 'rb') as f:")
        print(f"       model = pickle.load(f)")
    else:
        print("⚠ Model not found. Make sure training completed successfully.")
else:
    print("Running locally - model already saved in project directory")

## 📊 Performance Summary

Your model's performance on the validation/dev set (this is your DER for the project report).

In [ ]:
import pickle

# Load the trained model to get evaluation metrics
model_path = '/kaggle/working/models/logreg_model.pkl' if IS_KAGGLE else '../models/logreg_model.pkl'

if os.path.exists(model_path):
    print("="*70)
    print(" "*20 + "📊 FINAL RESULTS 📊")
    print("="*70)
    
    # Load model to re-evaluate if needed
    with open(model_path, 'rb') as f:
        trained_model = pickle.load(f)
    
    print(f"\n✓ Model Type: Logistic Regression (From Scratch)")
    print(f"✓ Training Samples: {sample_size if 'sample_size' in locals() else 'N/A'}")
    print(f"✓ Features: TF-IDF (window-based context)")
    print(f"✓ Model Size: {os.path.getsize(model_path) / (1024 * 1024):.2f} MB")
    
    print("\n" + "="*70)
    print(" "*15 + "📈 DEV SET PERFORMANCE (Your DER)")
    print("="*70)
    
    # The DER was already calculated during training
    # For clarity, let's re-display it prominently
    print("\n⚠ IMPORTANT: Check the 'Dev Set Results' section above for:")
    print("   • Accuracy: Percentage of correct diacritic predictions")
    print("   • DER (Diacritic Error Rate): 1 - Accuracy")
    print("   • This DER value is what you report in your project document")
    
    print("\n💡 Understanding Your Results:")
    print("   • DER < 0.10 (< 10%): Excellent")
    print("   • DER 0.10-0.20 (10-20%): Very Good")
    print("   • DER 0.20-0.30 (20-30%): Good")
    print("   • DER 0.30-0.50 (30-50%): Acceptable (baseline models)")
    print("   • DER > 0.50 (> 50%): Needs improvement")
    
    print("\n📝 For Your Project Report:")
    print("   1. Record the DER from 'Dev Set Results' above")
    print("   2. Compare with other models (CRF, etc.)")
    print("   3. Explain which features/parameters you tried")
    print("   4. Justify your final model choice for test set")
    
    print("\n" + "="*70)
else:
    print("⚠ Model file not found. Training may have failed.")

## Verify Submission File

Let's check the first few lines of the submission file.

In [ ]:
import csv

print("Submission file preview:")
print("-" * 80)

with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    for i, row in enumerate(reader):
        if i < 10:  # Show first 10 lines
            print(f"{row[0]}: {row[1][:50]}..." if len(row[1]) > 50 else f"{row[0]}: {row[1]}")
        else:
            break

print("-" * 80)
print("✓ Submission file is ready for upload!")

## Model Details

### Architecture
- **Feature Extraction**: Character-level TF-IDF with n-grams (1-3)
- **Context**: Sliding window of size 5 (captures ±2 characters)
- **Classifier**: Softmax regression (multinomial logistic regression)
- **Optimization**: Mini-batch gradient descent (batch size: 64)
- **Regularization**: L2 penalty (λ = 0.01)

### Parameters
- Learning rate: 0.01
- Max iterations: 500
- Max features: 5000
- N-gram range: (1, 3)

### Implementation
All code is written from scratch using only NumPy:
- Custom TF-IDF vectorizer
- Manual softmax computation with numerical stability
- Gradient descent with mini-batching
- Cross-entropy loss with regularization

**No external ML libraries used!**